In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import json
jsonPath = "/content/drive/MyDrive/dev/GeoguessrAI/aitestWithEU.json"

with open(jsonPath, 'r') as file:
    data = json.load(file)


print(data["customCoordinates"][0]["lat"])


blocksCreated=0
avgLocsPerBlock=0
blockSize=250 #this will act as threshold value

#Uses recursion making smaller and smaller areas until the number of locations in a block in under a certain threshold
def createBlock(isSqaure, startLat, endLat, startLong, endLong):
  global blocksCreated, avgLocsPerBlock
  numOfLocs=0
  print(f"Lat: {startLat} to {endLat}, Long: {startLong} to {endLong}")
  for loc in data["customCoordinates"]:
    if loc["lat"] < endLat and loc["lat"] > startLat and loc["lng"] < endLong and loc["lng"] > startLong:
      loc["block"] = {
            "startLat": startLat,
            "endLat": endLat,
            "startLong": startLong,
            "endLong": endLong
        }
      numOfLocs+=1
    # split if too many locs or block size is really big
    if numOfLocs>blockSize or endLong-startLong >7:
      if isSqaure:
        createBlock(False, startLat, (endLat+startLat)/2, startLong, endLong)
        createBlock(False, (endLat+startLat)/2, endLat, startLong, endLong)
      else:
        createBlock(True, startLat, endLat, startLong, (endLong+startLong)/2)
        createBlock(True, startLat, endLat, (endLong+startLong)/2, endLong)
      return
  if numOfLocs>0:
    print(f"Lat: {startLat} to {endLat}, Long: {startLong} to {endLong} created with {numOfLocs} locs.")
    blocksCreated+=1
    avgLocsPerBlock=(avgLocsPerBlock*(blocksCreated-1)+numOfLocs)/blocksCreated
    # print(avgLocsPerBlock)
  return


createBlock(True, -90, 90, -180, 180)

Streaming output truncated to the last 5000 lines.
Lat: 8.4375 to 11.25, Long: 16.875 to 22.5
Lat: 0.0 to 11.25, Long: 22.5 to 45.0
Lat: 0.0 to 5.625, Long: 22.5 to 45.0
Lat: 0.0 to 5.625, Long: 22.5 to 33.75
Lat: 0.0 to 2.8125, Long: 22.5 to 33.75
Lat: 0.0 to 2.8125, Long: 22.5 to 28.125
Lat: 0.0 to 2.8125, Long: 28.125 to 33.75
Lat: 0.0 to 1.40625, Long: 28.125 to 33.75
Lat: 0.0 to 1.40625, Long: 28.125 to 30.9375
Lat: 0.0 to 1.40625, Long: 28.125 to 30.9375 created with 8 locs.
Lat: 0.0 to 1.40625, Long: 30.9375 to 33.75
Lat: 0.0 to 0.703125, Long: 30.9375 to 33.75
Lat: 0.0 to 0.703125, Long: 30.9375 to 32.34375
Lat: 0.0 to 0.703125, Long: 32.34375 to 33.75
Lat: 0.0 to 0.3515625, Long: 32.34375 to 33.75
Lat: 0.0 to 0.3515625, Long: 32.34375 to 33.75 created with 202 locs.
Lat: 0.3515625 to 0.703125, Long: 32.34375 to 33.75
Lat: 0.3515625 to 0.703125, Long: 32.34375 to 33.75 created with 105 locs.
Lat: 0.703125 to 1.40625, Long: 30.9375 to 33.75
Lat: 1.40625 to 2.8125, Long: 28.125 t

In [ ]:
#verify by printing some sample data
for i in range(0,10):
  print(data["customCoordinates"][i])

#block metrics
print(avgLocsPerBlock)
print(blocksCreated)

{'lat': 39.91779852231334, 'lng': 20.178439616003498, 'heading': 286.1768, 'pitch': 0, 'zoom': 0, 'panoId': 'XiAoXAo3n32F-zlA81PCgQ', 'countryCode': None, 'stateCode': None, 'extra': {'panoDate': '2016-06'}, 'block': {'startLat': 39.7265625, 'endLat': 40.078125, 'startLong': 19.6875, 'endLong': 21.09375}}
{'lat': 42.2509938655425, 'lng': 20.270841820048737, 'heading': 201.62726, 'pitch': 0, 'zoom': 0, 'panoId': '_Z4metv4Hr5aOWkhKDk4FA', 'countryCode': None, 'stateCode': None, 'extra': {'panoDate': '2016-06'}, 'block': {'startLat': 42.1875, 'endLat': 42.890625, 'startLong': 19.6875, 'endLong': 22.5}}
{'lat': 41.64273962374177, 'lng': 19.71213327354392, 'heading': 144.39789, 'pitch': 0, 'zoom': 0, 'panoId': 'qYp3zps-LVzjxXoAA-ZF4g', 'countryCode': None, 'stateCode': None, 'extra': {'panoDate': '2016-06'}, 'block': {'startLat': 41.484375, 'endLat': 42.1875, 'startLong': 19.6875, 'endLong': 21.09375}}
{'lat': 41.34630367847824, 'lng': 19.753245058352604, 'heading': 315.83438, 'pitch': 0, '

In [ ]:
with open("output.json", "w") as f:
    json.dump(data, f, indent=2)

In [ ]:
import json

input_file = "output.json"
output_file = "blocks_only.json"

with open(input_file, "r") as f:
    data = json.load(f)

# Use a set to remove duplicates
unique_blocks = []
seen = set()

for coord in data.get("customCoordinates", []):
    block = coord.get("block")
    if block:
        # Turn block dict into a sorted tuple of items
        block_tuple = tuple(sorted(block.items()))
        if block_tuple not in seen:
            seen.add(block_tuple)
            unique_blocks.append(block)

# Save result
with open(output_file, "w") as f:
    json.dump(unique_blocks, f, indent=2)

print(f"Extracted {len(unique_blocks)} unique blocks to {output_file}")


Extracted 2111 unique blocks to blocks_only.json


In [ ]:
#create labels csv
import csv
csv_data =["filename", "labels"]

with open("labels.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(csv_data)
    for loc in data["customCoordinates"]:

      label_string = f"{loc['block']['startLat']}_{loc['block']['endLat']}_{loc['block']['startLong']}_{loc['block']['endLong']}"
      csv_data=[f"{loc['panoId']} - ", label_string]

      writer.writerow(csv_data)
